# Aktywacja Strategii Odbicie Ogórkowe
Ten notatnik sĹ‚uĹĽy do testowania, wizualizacji i optymalizacji strategii powrotu do Ĺ›redniej po mocnych spadkach.

## Importy, dane i sygnały

In [118]:
import os
import sys

# Dodajemy folder glowny do path aby moduly dzialaly
sys.path.append(os.path.abspath('c:/Users/PC/Documents/Antigravity/lyse-lby'))
os.chdir('c:/Users/PC/Documents/Antigravity/lyse-lby')

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Importujemy ladowanie danych i nasze nowe moduly
from core.ladowanie_danych import create_stock_dfs
from odbicie.mackowe_sygnaly import mackowe_sygnaly
from odbicie.odbicie import generate_odbicie_entries
from odbicie.odbicie_atr import generate_odbicie_atr_entries
from odbicie.odbicie_bb import generate_odbicie_bb_entries
from odbicie.tbm import moving_triple_barrier_labels
from odbicie.optymalizacja import optimize_atr_tbm, optimize_bb_tbm

# Katalog na pliki cache - zawsze wewnatrz odbicie/dane/, niezalezny od cwd
# __vsc_ipynb_file__ dostepny w VS Code; fallback na abspath('')
_SCRIPT_DIR = os.path.dirname(os.path.abspath(globals()["__vsc_ipynb_file__"])) if "__vsc_ipynb_file__" in globals() else os.path.abspath("")
# Jesli notebook lezy wewnatrz odbicie/, katalog jest katalogiem rodzica
if os.path.basename(_SCRIPT_DIR) != 'odbicie':
    _SCRIPT_DIR = os.path.join(_SCRIPT_DIR, 'odbicie')
DANE_DIR = os.path.join(_SCRIPT_DIR, 'cache')
os.makedirs(DANE_DIR, exist_ok=True)
print(f"Cache dir: {DANE_DIR}")

# Ustawienia ladowania danych
import json
with open(os.path.join(_SCRIPT_DIR, 'settings.json'), 'r') as f:
    all_settings = json.load(f)

# Ustawienia ladowania danych
settings = all_settings['data_settings']

Cache dir: c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache


In [119]:
# 1. Ladowanie Danych
import pickle

market = settings.get('market', 'all')
data_cache_file = os.path.join(DANE_DIR, f'dfs_cache_{market}.pkl')

if os.path.exists(data_cache_file):
    print("Znaleziono zapisane dane. Wczytywanie z pliku...")
    with open(data_cache_file, "rb") as f:
        dfs_1d, dfs_1w = pickle.load(f)
    print(f"Wczytano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W z pliku {data_cache_file}.")
else:
    print("Ladowanie danych dziennych i tygodniowych...")
    dfs_1d, dfs_1w = create_stock_dfs(settings)
    print(f"Zaladowano {len(dfs_1d)} symboli 1D i {len(dfs_1w)} symboli 1W.")
    print("Zapisywanie danych do pliku...")
    with open(data_cache_file, "wb") as f:
        pickle.dump((dfs_1d, dfs_1w), f)
    print("Dane zapisane pomyslnie.")


Znaleziono zapisane dane. Wczytywanie z pliku...
Wczytano 478 symboli 1D i 478 symboli 1W z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache\dfs_cache_sp500.pkl.


In [120]:
# 2. Generowanie Sygnalow Bazowych (mackowe_sygnaly)
market = settings.get('market', 'all')
signals_cache_file = os.path.join(DANE_DIR, f'signals_cache_{market}.pkl')

if os.path.exists(signals_cache_file):
    print("Znaleziono zapisane sygnaly. Wczytywanie z pliku...")
    with open(signals_cache_file, "rb") as f:
        signals_df = pickle.load(f)
    print(f"Wczytano {len(signals_df)} sygnalow z pliku {signals_cache_file}.")
else:
    signals_df = mackowe_sygnaly(
        dfs=dfs_1w,
        settings=settings,
        require_vol_confirmation=True,
        require_cmo_confirmation=True,
        interval='1w',
        entry_offset=0,
        pattern_cols=['hammer', 'inverted_hammer', 'engulfing_bull', 'piercing_line'],
        debug=True
    )
    print("Zapisywanie sygnalow do pliku...")
    with open(signals_cache_file, "wb") as f:
        pickle.dump(signals_df, f)
    print("Sygnaly zapisane pomyslnie.")

signals_df.describe()


Znaleziono zapisane sygnaly. Wczytywanie z pliku...
Wczytano 912 sygnalow z pliku c:\Users\PC\Documents\Antigravity\lyse-lby\odbicie\cache\signals_cache_sp500.pkl.


,signal_time,entry_time,signal_close
count,912,912,912.000000
mean,2023-10-08 09:01:34.736842,2023-10-08 09:01:34.736842,129.103488
min,2021-07-25 00:00:00,2021-07-25 00:00:00,7.970000
25%,2022-05-15 00:00:00,2022-05-15 00:00:00,51.740002
50%,2023-10-08 00:00:00,2023-10-08 00:00:00,97.340000
75%,2025-03-16 00:00:00,2025-03-16 00:00:00,179.127495
max,2026-02-22 00:00:00,2026-02-22 00:00:00,565.369995
std,NaN,NaN,103.256636


## Wejście i Wyjście

In [121]:
# 3. Wybor Strategii Wejscia
# Zmien ta zmienna aby przełaczyc strategie: 'base', 'atr', 'bb'
STRATEGY = 'base'

strat_settings = all_settings['strategies'][STRATEGY]['entry_settings']
tbm_settings = all_settings['strategies'][STRATEGY]['tbm_settings']

if STRATEGY == 'base':
    # --- Strategia bazowa: staly prog procentowy ---
    threshold_pct = strat_settings['threshold_pct']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        max_setup_hold_bars=max_hold
    )
    print(f"[base] Wygenerowano {len(entries_df)} wejsc przy progu {threshold_pct*100}%")

elif STRATEGY == 'atr':
    # --- Strategia ATR: prog oparty na wielokrotnosci ATR ---
    atr_period = strat_settings['atr_period']
    atr_factor = strat_settings['atr_factor']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        max_setup_hold_bars=max_hold
    )
    print(f"[atr] Wygenerowano {len(entries_df)} wejsc (period={atr_period}, factor={atr_factor})")

elif STRATEGY == 'bb':
    # --- Strategia BB: wejscie przy dotknięciu dolnej wstegi Bollingera ---
    bb_period = strat_settings['bb_period']
    bb_std   = strat_settings['bb_std']
    max_hold = strat_settings['max_setup_hold_bars']
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        max_setup_hold_bars=max_hold
    )
    print(f"[bb] Wygenerowano {len(entries_df)} wejsc (period={bb_period}, std={bb_std})")

else:
    raise ValueError(f"Nieznana strategia: '{STRATEGY}'. Uzyj 'base', 'atr' lub 'bb'.")

entries_df.head()


[base] Wygenerowano 34 wejsc przy progu 15.0%


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr
0,APP,2022-11-13,hammer,2022-11-16,14.399000,16.940001,0.15,1.428259
1,APP,2022-05-15,engulfing_bull,2022-05-24,33.540999,39.459999,0.15,4.059080
2,XYZ,2022-01-16,inverted_hammer,2022-01-24,113.296494,133.289993,0.15,9.582179
3,XYZ,2022-02-27,engulfing_bull,2022-03-07,101.847000,119.820000,0.15,10.631999
4,CCL,2022-05-15,hammer,2022-05-24,12.138000,14.280000,0.15,1.023463


In [122]:
# 4. Wyjście z użyciem Moving Triple Barrier Method
trades_df = moving_triple_barrier_labels(
    entries_df=entries_df,
    market_data_daily=dfs_1d,
    tp_mult=tbm_settings['tpm'],
    sl_mult=tbm_settings['slm'],
    tp_trail_mult=tbm_settings['ttpm'],
    max_holding_bars=tbm_settings['mhb'],
    early_breakeven=tbm_settings['early_bailout'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    max_loss_pct=tbm_settings['max_loss_pct'],
    exit_on_close=tbm_settings.get('exit_on_close', True)
)
print(f"Zakończono {len(trades_df)} transakcji.")
trades_df.head()


Zakończono 34 transakcji.


,symbol,signal_time,pattern,entry_time,entry_price,signal_close,threshold_pct,entry_atr,exit_time,exit_price,return_pct,exit_reason,hold_bars
0,APP,2022-11-13,hammer,2022-11-16,14.399000,16.940001,0.15,1.428259,2022-11-21,13.300000,-7.632476,TRAILING_SL,3
1,APP,2022-05-15,engulfing_bull,2022-05-24,33.540999,39.459999,0.15,4.059080,2022-05-31,38.110001,13.622139,TRAILING_TP,4
2,XYZ,2022-01-16,inverted_hammer,2022-01-24,113.296494,133.289993,0.15,9.582179,2022-01-26,111.000000,-2.026977,TRAILING_TP,2
3,XYZ,2022-02-27,engulfing_bull,2022-03-07,101.847000,119.820000,0.15,10.631999,2022-03-10,108.870003,6.895641,TRAILING_TP,3
4,CCL,2022-05-15,hammer,2022-05-24,12.138000,14.280000,0.15,1.023463,2022-06-01,13.470000,10.973806,TRAILING_TP,5


## Analiza

In [123]:
# 5. Analiza i Statystyki
if not trades_df.empty:
    wins = (trades_df['return_pct'] > 0).sum()
    losses = (trades_df['return_pct'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    trades_df['return_per_bar'] = trades_df['return_pct'] / trades_df['hold_bars']
    
    print(f"Total Trades: {len(trades_df)}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Avg Return: {trades_df['return_pct'].mean():.2f}%")
    print(f"Avg bars held: {trades_df['hold_bars'].mean():.2f}")
    print(f"Avg Return per Bar: {trades_df['return_per_bar'].mean():.2f}%")

    
    # Powody wyjĹ›cia
    print("\nExit Reasons:")
    print(trades_df['exit_reason'].value_counts())
else:
    print("Brak transakcji do analizy.")

Total Trades: 34
Win Rate: 58.82%
Avg Return: 0.28%
Avg bars held: 3.00
Avg Return per Bar: -0.43%

Exit Reasons:
exit_reason
TRAILING_TP    26
TRAILING_SL     7
SL              1
Name: count, dtype: int64


In [124]:
temp = pd.DataFrame({
    'count': trades_df.groupby('exit_reason').return_pct.count(),
    'avg_return': trades_df.groupby('exit_reason').return_pct.mean(),
    'cumulativ_return': trades_df.groupby('exit_reason').return_pct.sum(),
    'std': trades_df.groupby('exit_reason').return_pct.std(),
    'avg_hold_bars': trades_df.groupby('exit_reason').hold_bars.mean(),
    'std_hold_bars': trades_df.groupby('exit_reason').hold_bars.std(),
    'max_hold_bars': trades_df.groupby('exit_reason').hold_bars.max()
    })
    
temp

,count,avg_return,cumulativ_return,std,avg_hold_bars,std_hold_bars,max_hold_bars
exit_reason,,,,,,,
SL,1,-19.928757,-19.928757,NaN,1.000000,NaN,1
TRAILING_SL,7,-13.833281,-96.832969,5.773687,3.571429,2.439750,8
TRAILING_TP,26,4.856827,126.277493,6.060026,2.923077,1.230385,7


## Ploty

In [125]:
# 6. Interaktywna Wizualizacja Transakcji
from odbicie.plot import show_trade_viewer
import ipywidgets as widgets
from IPython.display import display, clear_output

# Inicjalny rysunek
show_trade_viewer(
    trades_df,
    dfs_1d,
    tpm=tbm_settings['tpm'],
    slm=tbm_settings['slm'],
    ttpm=tbm_settings['ttpm'],
    mhb=tbm_settings['mhb'],
    exit_reason='All',
    active_trailing_sl=tbm_settings['active_trail_sl'],
    sl_trail_mult=tbm_settings['sl_trail_mult'],
    max_loss_pct=tbm_settings['max_loss_pct'],
    time_decay_sl=tbm_settings['time_decay_sl'],
    exit_on_close=tbm_settings.get('exit_on_close', True),
    strategy_type="tbm"
)


Output()

## Optymalizacja

### ATR + TBM

In [126]:
# Optymalizacja wejsc ATR + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.

'''
atr_tbm_settings = all_settings['strategies']['atr']['tbm_settings']
atr_opt_results = optimize_atr_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    atr_periods=[15],
    atr_factors=[4.0],
    max_setup_hold_bars_list=[10],
    tp_mults=[0.8],
    sl_mults=[1.75],
    tp_trail_mults=[0.07],
    max_holding_bars_list=[7],
    early_breakeven=atr_tbm_settings['early_bailout'],
    time_decay_sl=atr_tbm_settings['time_decay_sl'],
    active_trailing_sl=atr_tbm_settings['active_trail_sl'],
    sl_trail_mult=atr_tbm_settings['sl_trail_mult'],
    max_loss_pct=atr_tbm_settings['max_loss_pct'],
    exit_on_close=atr_tbm_settings.get('exit_on_close', True),
    min_trades=10
)
display(atr_opt_results.head(20))
'''


"\natr_tbm_settings = all_settings['strategies']['atr']['tbm_settings']\natr_opt_results = optimize_atr_tbm(\n    signals_df=signals_df,\n    market_data_daily=dfs_1d,\n    atr_periods=[15],\n    atr_factors=[4.0],\n    max_setup_hold_bars_list=[10],\n    tp_mults=[0.8],\n    sl_mults=[1.75],\n    tp_trail_mults=[0.07],\n    max_holding_bars_list=[7],\n    early_breakeven=atr_tbm_settings['early_bailout'],\n    time_decay_sl=atr_tbm_settings['time_decay_sl'],\n    active_trailing_sl=atr_tbm_settings['active_trail_sl'],\n    sl_trail_mult=atr_tbm_settings['sl_trail_mult'],\n    max_loss_pct=atr_tbm_settings['max_loss_pct'],\n    exit_on_close=atr_tbm_settings.get('exit_on_close', True),\n    min_trades=10\n)\ndisplay(atr_opt_results.head(20))\n"

### BB + TBM

In [127]:
# Optymalizacja wejsc BB + wyjsc TBM
# Odkomentuj blok ponizej, aby uruchomic optymalizacje.

'''
bb_tbm_settings = all_settings['strategies']['bb']['tbm_settings']
bb_opt_results = optimize_bb_tbm(
    signals_df=signals_df,
    market_data_daily=dfs_1d,
    bb_periods=[10],
    bb_stds=[3.0],
    max_setup_hold_bars_list=[10],
    tp_mults=[0.2],
    sl_mults=[2.0],
    tp_trail_mults=[0.04],
    max_holding_bars_list=[7],
    early_breakeven=bb_tbm_settings['early_bailout'],
    time_decay_sl=bb_tbm_settings['time_decay_sl'],
    active_trailing_sl=bb_tbm_settings['active_trail_sl'],
    sl_trail_mult=bb_tbm_settings['sl_trail_mult'],
    max_loss_pct=bb_tbm_settings['max_loss_pct'],
    exit_on_close=bb_tbm_settings.get('exit_on_close', True),
    min_trades=10
)
display(bb_opt_results.head(20))
'''


"\nbb_tbm_settings = all_settings['strategies']['bb']['tbm_settings']\nbb_opt_results = optimize_bb_tbm(\n    signals_df=signals_df,\n    market_data_daily=dfs_1d,\n    bb_periods=[10],\n    bb_stds=[3.0],\n    max_setup_hold_bars_list=[10],\n    tp_mults=[0.2],\n    sl_mults=[2.0],\n    tp_trail_mults=[0.04],\n    max_holding_bars_list=[7],\n    early_breakeven=bb_tbm_settings['early_bailout'],\n    time_decay_sl=bb_tbm_settings['time_decay_sl'],\n    active_trailing_sl=bb_tbm_settings['active_trail_sl'],\n    sl_trail_mult=bb_tbm_settings['sl_trail_mult'],\n    max_loss_pct=bb_tbm_settings['max_loss_pct'],\n    exit_on_close=bb_tbm_settings.get('exit_on_close', True),\n    min_trades=10\n)\ndisplay(bb_opt_results.head(20))\n"

### Optuna => BB

In [128]:
def objective_bb_tbm_direct(trial):
    # Optymalizacja Parametrow BB
    bb_period = trial.suggest_int("bb_period", 5, 20)
    bb_std = trial.suggest_float("bb_std", 1.0, 4.0, step=0.1)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 5, 15)
    
    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.1, 2.0, step=0.1)
    sl_mult = trial.suggest_float("sl_mult", 0.5, 3.0, step=0.1)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0, step=0.5)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 20)
    
    bb_tbm_settings = all_settings['strategies']['bb']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_bb_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        bb_period=bb_period,
        bb_std=bb_std,
        rsi_period=14,
        max_setup_hold_bars=max_setup_hold_bars,
        buy_on_close=False
    )
    
    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        early_breakeven=bb_tbm_settings['early_bailout'],
        time_decay_sl=time_decay_sl,
        active_trailing_sl=bb_tbm_settings['active_trail_sl'],
        sl_trail_mult=sl_trail_mult,
        max_loss_pct=bb_tbm_settings['max_loss_pct'],
        exit_on_close=bb_tbm_settings.get('exit_on_close', True)
    )
    
    if len(trds) < 10:
        return 0.0
        
    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0
        
    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar

'''
study_bb_tbm_direct = optuna.create_study(direction="maximize")
study_bb_tbm_direct.optimize(objective_bb_tbm_direct, n_trials=50)

print("Best parameters:", study_bb_tbm_direct.best_params)
print("Best value:", study_bb_tbm_direct.best_value)
'''

'\nstudy_bb_tbm_direct = optuna.create_study(direction="maximize")\nstudy_bb_tbm_direct.optimize(objective_bb_tbm_direct, n_trials=50)\n\nprint("Best parameters:", study_bb_tbm_direct.best_params)\nprint("Best value:", study_bb_tbm_direct.best_value)\n'

### Optuna => ATR


In [129]:
def objective_atr_tbm_direct(trial):
    # Optymalizacja Parametrow ATR
    atr_period = trial.suggest_int("atr_period", 15)
    atr_factor = trial.suggest_float("atr_factor", 4.0)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 7)
    
    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.1, 2.0, step=0.1)
    sl_mult = trial.suggest_float("sl_mult", 0.5, 3.0, step=0.1)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0, step=0.5)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 20)
    
    atr_tbm_settings = all_settings['strategies']['atr']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_atr_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        atr_period=atr_period,
        atr_factor=atr_factor,
        max_setup_hold_bars=max_setup_hold_bars,
        buy_on_close=False
    )
    
    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        early_breakeven=atr_tbm_settings['early_bailout'],
        time_decay_sl=time_decay_sl,
        active_trailing_sl=atr_tbm_settings['active_trail_sl'],
        sl_trail_mult=sl_trail_mult,
        max_loss_pct=atr_tbm_settings['max_loss_pct'],
        exit_on_close=atr_tbm_settings.get('exit_on_close', True)
    )
    
    if len(trds) < 10:
        return 0.0
        
    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0
        
    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar

'''
study_atr_tbm_direct = optuna.create_study(direction="maximize")
study_atr_tbm_direct.optimize(objective_atr_tbm_direct, n_trials=50)

print("Best parameters:", study_atr_tbm_direct.best_params)
print("Best value:", study_atr_tbm_direct.best_value)
'''


'\nstudy_atr_tbm_direct = optuna.create_study(direction="maximize")\nstudy_atr_tbm_direct.optimize(objective_atr_tbm_direct, n_trials=50)\n\nprint("Best parameters:", study_atr_tbm_direct.best_params)\nprint("Best value:", study_atr_tbm_direct.best_value)\n'

### Optuna => base

In [132]:
def objective_base_tbm_direct(trial):
    # Optymalizacja Parametrow base
    threshold_pct = trial.suggest_float("threshold_pct", 0.18, 0.18)
    max_setup_hold_bars = trial.suggest_int("max_setup_hold_bars", 5, 15)
    
    # Optymalizacja Wyjsc (TBM)
    tp_mult = trial.suggest_float("tp_mult", 0.1, 2.0, step=0.1)
    sl_mult = trial.suggest_float("sl_mult", 0.5, 3.0, step=0.1)
    tp_trail_mult = trial.suggest_float("tp_trail_mult", 0.01, 0.2, step=0.01)
    sl_trail_mult = trial.suggest_float("sl_trail_mult", 0.5, 5.0, step=0.5)
    time_decay_sl = trial.suggest_categorical("time_decay_sl", [True, False])
    max_holding_bars = trial.suggest_int("max_holding_bars", 5, 20)
    
    base_tbm_settings = all_settings['strategies']['base']['tbm_settings']

    # 1. Generowanie wejsc
    entries_df = generate_odbicie_entries(
        signals_df=signals_df,
        market_data_daily=dfs_1d,
        threshold_pct=threshold_pct,
        max_setup_hold_bars=max_setup_hold_bars,
        buy_on_close=False
    )
    
    if entries_df.empty or len(entries_df) < 10:
        return 0.0

    # 2. Ewaluacja TBM
    trds = moving_triple_barrier_labels(
        entries_df=entries_df,
        market_data_daily=dfs_1d,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        tp_trail_mult=tp_trail_mult,
        max_holding_bars=max_holding_bars,
        early_breakeven=base_tbm_settings['early_bailout'],
        time_decay_sl=time_decay_sl,
        active_trailing_sl=base_tbm_settings['active_trail_sl'],
        sl_trail_mult=sl_trail_mult,
        max_loss_pct=base_tbm_settings['max_loss_pct'],
        exit_on_close=base_tbm_settings.get('exit_on_close', True)
    )
    
    if len(trds) < 10:
        return 0.0
        
    avg_hold_bars = trds['hold_bars'].mean()
    if avg_hold_bars == 0:
        return 0.0
        
    avg_return = trds['return_pct'].mean()
    return avg_return / avg_hold_bars  # return_per_bar

'''
study_base_tbm_direct = optuna.create_study(direction="maximize")
study_base_tbm_direct.optimize(objective_base_tbm_direct, n_trials=50)

print("Best parameters:", study_base_tbm_direct.best_params)
print("Best value:", study_base_tbm_direct.best_value)
'''

[I 2026-03-22 19:44:37,625] A new study created in memory with name: no-name-12bc55b3-4cc7-406b-952c-cb5f4e199e95
[I 2026-03-22 19:44:40,317] Trial 0 finished with value: 1.2141119416967834 and parameters: {'threshold_pct': 0.18, 'max_setup_hold_bars': 13, 'tp_mult': 0.30000000000000004, 'sl_mult': 2.1, 'tp_trail_mult': 0.13, 'sl_trail_mult': 2.5, 'time_decay_sl': True, 'max_holding_bars': 19}. Best is trial 0 with value: 1.2141119416967834.
[I 2026-03-22 19:44:43,305] Trial 1 finished with value: 0.9517087423085115 and parameters: {'threshold_pct': 0.18, 'max_setup_hold_bars': 8, 'tp_mult': 1.9000000000000001, 'sl_mult': 0.9, 'tp_trail_mult': 0.09999999999999999, 'sl_trail_mult': 4.5, 'time_decay_sl': False, 'max_holding_bars': 12}. Best is trial 0 with value: 1.2141119416967834.
[I 2026-03-22 19:44:45,528] Trial 2 finished with value: 0.8094203205638751 and parameters: {'threshold_pct': 0.18, 'max_setup_hold_bars': 7, 'tp_mult': 1.7000000000000002, 'sl_mult': 0.9, 'tp_trail_mult': 0.

Best parameters: {'threshold_pct': 0.18, 'max_setup_hold_bars': 7, 'tp_mult': 0.2, 'sl_mult': 1.7000000000000002, 'tp_trail_mult': 0.15000000000000002, 'sl_trail_mult': 0.5, 'time_decay_sl': True, 'max_holding_bars': 8}
Best value: 1.4507318842191632
